# 06 - 指标计算

**目标**：从生成的 GeoJSON 图层计算 haidian 所需的所有空间指标。

**指标清单**：
| 指标 | 公式 | 单位 |
|------|------|------|
| 容积率 (FAR) | 总建筑面积 / 总用地面积 | 无量纲 |
| 建筑密度 | 建筑基底面积 / 总用地面积 | % |
| 绿地率 | 绿地面积 / 总用地面积 | % |
| 道路网密度 | 道路总长度 / 总用地面积 | km/km² |

**关键原则**：每个指标都可追溯到源几何数据。

In [1]:
import sys
sys.path.insert(0, '..')

import geopandas as gpd
import numpy as np
import json
from shapely.ops import unary_union

from src.projection import (
    transform_geometry,
    compute_area_4548,
    compute_area_by_group,
    CRS_4326,
    CRS_4548,
)
from src.generation import (
    subdivide_boundary,
    assign_land_use,
    generate_buildings_in_parcel,
    LAND_USE_LABELS,
)
from src.topology import check_coverage, check_overlaps, check_containment

print('Libraries loaded successfully.')

Libraries loaded successfully.


## Step 1: 重新生成所有数据层

确保每个指标都有可追溯的源数据。

In [2]:
# Load and transform boundary
gdf_all = gpd.read_file('../data/haidian-boundary.geojson')
site = gdf_all[gdf_all['id'] == 'PROV-SITE-001'].iloc[0]
boundary_4548 = transform_geometry(site.geometry, CRS_4326, CRS_4548)

# Generate land use parcels
cells = subdivide_boundary(boundary_4548, n_rows=20, n_cols=15)
land_use_gdf = assign_land_use(cells, random_seed=42)
land_use_gdf = compute_area_4548(land_use_gdf.to_crs(CRS_4548))

# Ensure consistent CRS
land_use_gdf = land_use_gdf.to_crs(CRS_4548)

# Total site area
total_site_area_sqm = boundary_4548.area
total_site_area_ha = total_site_area_sqm / 10_000

print(f'Total site area: {total_site_area_sqm:,.0f} sqm = {total_site_area_ha:.2f} ha')
print(f'Land use parcels: {len(land_use_gdf)}')

Total site area: 11,412,825 sqm = 1141.28 ha
Land use parcels: 278


## Step 2: 建筑面积估算 (用于 FAR 计算)

建筑基底面积 × 平均层数 ≈ 总建筑面积

不同用地类型的平均层数不同。

In [3]:
# Average floors by land use type
AVG_FLOORS = {
    'R': 12,   # Residential — 12 floors
    'A': 8,    # Administration — 8 floors
    'B': 10,   # Commercial — 10 floors
    'G': 1,    # Green space — 1 floor (pavilions)
    'S': 2,    # Transportation — 2 floors
    'E': 5,    # Education — 5 floors
}

# Coverage ratios (from notebook 03)
COVERAGE_BY_LANDUSE = {
    'R': 0.30,
    'A': 0.25,
    'B': 0.35,
    'G': 0.05,
    'S': 0.10,
    'E': 0.20,
}

In [4]:
# Calculate metrics by land use type
area_by_lu = land_use_gdf.groupby('land_use_code')['area_sqm'].sum()

print('=== Land Use Area Breakdown ===')
print()
for lu in sorted(area_by_lu.index):
    area = area_by_lu[lu]
    pct = area / total_site_area_sqm * 100
    avg_floors = AVG_FLOORS.get(lu, 1)
    coverage = COVERAGE_BY_LANDUSE.get(lu, 0.25)
    bldg_footprint = area * coverage
    total_gfa = bldg_footprint * avg_floors  # Gross Floor Area
    label = LAND_USE_LABELS.get(lu, lu)
    print(f'{label:30s} Area: {area:10,.0f} sqm ({pct:5.1f}%) | GFA: {total_gfa:10,.0f} sqm')

=== Land Use Area Breakdown ===

A 公共管理与公共服务用地                  Area:    975,368 sqm (  8.5%) | GFA:  1,950,736 sqm
B 商业服务业设施用地                    Area:  1,259,337 sqm ( 11.0%) | GFA:  4,407,679 sqm
E 其他建设用地                       Area:  1,207,423 sqm ( 10.6%) | GFA:  1,207,423 sqm
G 绿地与广场用地                      Area:  2,061,655 sqm ( 18.1%) | GFA:    103,083 sqm
R 居住用地                         Area:  4,345,772 sqm ( 38.1%) | GFA: 15,644,780 sqm
S 道路与交通设施用地                    Area:  1,563,271 sqm ( 13.7%) | GFA:    312,654 sqm


## Step 3: 计算所有指标

In [5]:
# --- FAR (Floor Area Ratio) ---
total_gfa = 0  # Total Gross Floor Area
total_bldg_footprint = 0  # Total building footprint

for lu in area_by_lu.index:
    area = area_by_lu[lu]
    coverage = COVERAGE_BY_LANDUSE.get(lu, 0.25)
    avg_floors = AVG_FLOORS.get(lu, 1)
    bldg_fp = area * coverage
    total_bldg_footprint += bldg_fp
    total_gfa += bldg_fp * avg_floors

far = total_gfa / total_site_area_sqm

# --- Building Density ---
building_density = total_bldg_footprint / total_site_area_sqm * 100

# --- Green Ratio ---
green_area = area_by_lu.get('G', 0)
green_ratio = green_area / total_site_area_sqm * 100

# --- Road Network Density (from notebook 04 logic) ---
from shapely.geometry import LineString

def generate_grid_roads(boundary_poly, spacing_m):
    minx, miny, maxx, maxy = boundary_poly.bounds
    total_len = 0.0
    x = minx
    while x <= maxx:
        line = LineString([(x, miny), (x, maxy)])
        clipped = line.intersection(boundary_poly)
        total_len += clipped.length
        x += spacing_m
    y = miny
    while y <= maxy:
        line = LineString([(minx, y), (maxx, y)])
        clipped = line.intersection(boundary_poly)
        total_len += clipped.length
        y += spacing_m
    return total_len

# Compute total road length for all three classes
road_length_secondary = generate_grid_roads(boundary_4548, 1200)
road_length_branch = generate_grid_roads(boundary_4548, 600)
road_length_slow = generate_grid_roads(boundary_4548, 250)
total_road_length = road_length_secondary + road_length_branch + road_length_slow
road_density_km_per_km2 = (total_road_length / 1000) / (total_site_area_sqm / 1_000_000)

print('=' * 60)
print('  METRICS SUMMARY — Haidian AI Belt (Simulated Data)')
print('=' * 60)
print()
print(f'  Total Site Area:        {total_site_area_sqm:>12,.0f} sqm  = {total_site_area_ha:,.2f} ha')
print(f'  Total GFA (estimated):  {total_gfa:>12,.0f} sqm')
print(f'  Total Bldg Footprint:   {total_bldg_footprint:>12,.0f} sqm')
print()
print(f'  FAR (容积率):            {far:>12.4f}')
print(f'  Building Density (建筑密度): {building_density:>8.2f} %')
print(f'  Green Ratio (绿地率):     {green_ratio:>8.2f} %')
print(f'  Road Density (道路网密度):  {road_density_km_per_km2:>8.2f} km/km²')
print()
print(f'  Total Road Length:      {total_road_length:>12,.0f} m  = {total_road_length/1000:,.2f} km')

  METRICS SUMMARY — Haidian AI Belt (Simulated Data)

  Total Site Area:          11,412,825 sqm  = 1,141.28 ha
  Total GFA (estimated):    23,626,355 sqm
  Total Bldg Footprint:      2,489,236 sqm

  FAR (容积率):                  2.0702
  Building Density (建筑密度):    21.81 %
  Green Ratio (绿地率):        18.06 %
  Road Density (道路网密度):     12.65 km/km²

  Total Road Length:           144,379 m  = 144.38 km


## Step 4: 输出 metrics.json

与 haidian 提交规范兼容的格式。

In [6]:
metrics = {
    "meta": {
        "document": "metrics.json",
        "version": "1.0.0",
        "project": "haidian-ai-belt-provisional",
        "generated_by": "urban-spatial-tooling/notebooks/06-metrics-computation.ipynb",
        "crs": "EPSG:4548",
        "data_source": "simulated (provisional boundary from haidian project)",
        "note": "All metrics are computed from simulated data and are NOT authoritative."
    },
    "site": {
        "area_sqm": round(total_site_area_sqm, 2),
        "area_ha": round(total_site_area_ha, 4),
        "area_km2": round(total_site_area_sqm / 1_000_000, 6),
        "declared_area_sqm": int(site['area_sqm_declared']),
        "deviation_pct": round((total_site_area_sqm - site['area_sqm_declared']) / site['area_sqm_declared'] * 100, 4),
    },
    "land_use": {
        "total_parcels": len(land_use_gdf),
        "area_by_code": {
            lu: {
                "area_sqm": round(area_by_lu[lu], 2),
                "area_ha": round(area_by_lu[lu] / 10_000, 4),
                "pct": round(area_by_lu[lu] / total_site_area_sqm * 100, 2),
                "label_zh": LAND_USE_LABELS.get(lu, lu),
            }
            for lu in sorted(area_by_lu.index)
        },
    },
    "metrics": {
        "far": {
            "value": round(far, 4),
            "unit": "dimensionless",
            "formula": "total_gfa / total_site_area",
            "total_gfa_sqm": round(total_gfa, 2),
            "trace": "land_use_area * coverage_ratio * avg_floors, summed across all land_use_codes",
        },
        "building_density": {
            "value_pct": round(building_density, 2),
            "unit": "%",
            "formula": "total_building_footprint / total_site_area * 100",
            "total_footprint_sqm": round(total_bldg_footprint, 2),
            "trace": "sum(land_use_area * coverage_ratio) for each land_use_code",
        },
        "green_ratio": {
            "value_pct": round(green_ratio, 2),
            "unit": "%",
            "formula": "green_area / total_site_area * 100",
            "green_area_sqm": round(green_area, 2),
            "trace": "area_sqm of land_use_code='G'",
        },
        "road_density": {
            "value_km_per_km2": round(road_density_km_per_km2, 2),
            "unit": "km/km²",
            "formula": "total_road_length_km / total_site_area_km2",
            "total_road_length_m": round(total_road_length, 2),
            "trace": "total length of all grid road LineStrings clipped to boundary",
        },
    },
    "topology_checks": {
        "coverage": {
            "passed": check_coverage(land_use_gdf, boundary_4548)[0],
            "method": "unary_union.covers(boundary)",
        },
        "overlaps": {
            "passed": check_overlaps(land_use_gdf)[0],
            "method": "pairwise .intersection() for all parcel pairs",
        },
        "containment": {
            "passed": check_containment(land_use_gdf, boundary_4548)[0],
            "method": ".within(boundary) for all features",
        },
    },
}

# Write to file
output_path = '../outputs/metrics.json'
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print(f'Metrics written to: {output_path}')
print()
print('Top-level keys:', list(metrics.keys()))

Metrics written to: ../outputs/metrics.json

Top-level keys: ['meta', 'site', 'land_use', 'metrics', 'topology_checks']


In [7]:
# Quick verification: every metric is traceable
print('=== Traceability Verification ===')
for metric_name, metric_data in metrics['metrics'].items():
    print(f'{metric_name}: trace → {metric_data["trace"]}')

=== Traceability Verification ===
far: trace → land_use_area * coverage_ratio * avg_floors, summed across all land_use_codes
building_density: trace → sum(land_use_area * coverage_ratio) for each land_use_code
green_ratio: trace → area_sqm of land_use_code='G'
road_density: trace → total length of all grid road LineStrings clipped to boundary


## 总结

- 成功计算了所有四个核心指标（FAR、建筑密度、绿地率、道路网密度）
- 所有指标均可追溯到源几何数据
- 输出了符合 haidian 规范的 `metrics.json`
- 拓扑检查结果也包含在输出中
- 下一步：生成专业规划图纸

In [8]:
# Verify JSON file was written correctly
with open('../outputs/metrics.json', 'r', encoding='utf-8') as f:
    loaded = json.load(f)
print(f'Verification: FAR = {loaded["metrics"]["far"]["value"]}')
print(f'Verification: Green ratio = {loaded["metrics"]["green_ratio"]["value_pct"]}%')

Verification: FAR = 2.0702
Verification: Green ratio = 18.06%
